In [1]:
# 1. Import Libraries
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score



In [2]:
# 2. Loading the Dataset finding the shape and missing values per column 
df = pd.read_csv(r"D:\AI ML\diabetes12.csv")
print("Shape before preprocessing:", df.shape)
print("\nMissing values per column:\n", df.isnull().sum())


Shape before preprocessing: (300, 9)

Missing values per column:
 Pregnancies                  0
Glucose                     15
BloodPressure               15
SkinThickness               15
Insulin                     15
BMI                         15
DiabetesPedigreeFunction     0
Age                          0
Outcome                      0
dtype: int64


In [3]:
# 3. Handling Missing Values by median imputation 
for col in df.columns:
    if df[col].isnull().sum() > 0:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"Filled '{col}' with median = {median_val:.2f}")

print("\nMissing values after imputation:", df.isnull().sum().sum())

Filled 'Glucose' with median = 121.40
Filled 'BloodPressure' with median = 70.70
Filled 'SkinThickness' with median = 24.70
Filled 'Insulin' with median = 83.40
Filled 'BMI' with median = 27.90

Missing values after imputation: 0


In [4]:
# 4. Removing Outliers by using IQR Method
feature_cols = df.columns.drop("Outcome")

rows_before = df.shape[0]

for col in feature_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df = df[(df[col] >= lower) & (df[col] <= upper)]

rows_after = df.shape[0]

print("\nRows before:", rows_before)
print("Rows after :", rows_after)
print("Outliers removed:", rows_before - rows_after)

df.reset_index(drop=True, inplace=True)


Rows before: 300
Rows after : 285
Outliers removed: 15


In [5]:

# 5. Feature Scaling process through Standardization
X = df.drop("Outcome", axis=1)
y = df["Outcome"]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("\nFeature scaling done")
print("\nSample data:\n", X_scaled_df.head())


Feature scaling done

Sample data:
    Pregnancies   Glucose  BloodPressure  SkinThickness   Insulin       BMI  \
0     1.507287  0.061268      -0.960199       0.341299  0.692809  0.061350   
1    -0.651391  1.704363      -0.547525      -1.209680 -1.459328  0.315277   
2    -1.370950  0.061268      -0.493698      -0.304943 -0.134169  0.043212   
3     1.507287 -0.661010       0.277823      -1.048120 -0.600466 -1.607314   
4     0.068169 -0.948551      -0.879459       1.482991  2.338796 -1.552901   

   DiabetesPedigreeFunction       Age  
0                 -0.088213  0.345120  
1                  0.670491  0.412122  
2                 -0.191673  1.484156  
3                  0.670491 -0.391904  
4                 -1.295242 -1.195929  


In [6]:
# 6. Feature Selection by using Forward Selection Method
selected_features = []
remaining_features = list(X_scaled_df.columns)

while len(selected_features) < 4:
    best_feature = None
    best_score = 0

    for feature in remaining_features:
        temp_features = selected_features + [feature]
        X_temp = X_scaled_df[temp_features]

        X_train, X_test, y_train, y_test = train_test_split(
            X_temp, y, test_size=0.2, random_state=42
        )

        model = LogisticRegression(max_iter=1000)
        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)
        score = accuracy_score(y_test, y_pred)

        if score > best_score:
            best_score = score
            best_feature = feature

    selected_features.append(best_feature)
    remaining_features.remove(best_feature)

    print(f"Selected Feature: {best_feature}  Accuracy: {best_score:.4f}")

print("\nFinal Selected Features:", selected_features)

X_selected_df = X_scaled_df[selected_features]

Selected Feature: Glucose  Accuracy: 0.6667
Selected Feature: Pregnancies  Accuracy: 0.7018
Selected Feature: Age  Accuracy: 0.7193
Selected Feature: BloodPressure  Accuracy: 0.7193

Final Selected Features: ['Glucose', 'Pregnancies', 'Age', 'BloodPressure']


In [7]:
# 7 Final Preprocessed Dataset
df_preprocessed = X_selected_df.copy()
df_preprocessed["Outcome"] = y.values
print("\nFinal shape:", df_preprocessed.shape)
print("\nFinal sample data:\n", df_preprocessed.head())


Final shape: (285, 5)

Final sample data:
     Glucose  Pregnancies       Age  BloodPressure  Outcome
0  0.061268     1.507287  0.345120      -0.960199        1
1  1.704363    -0.651391  0.412122      -0.547525        1
2  0.061268    -1.370950  1.484156      -0.493698        0
3 -0.661010     1.507287 -0.391904       0.277823        0
4 -0.948551     0.068169 -1.195929      -0.879459        0


In [8]:
# 8. Saving the Preprocessed File 
df_preprocessed.to_csv(r"D:\AI ML\preprocessed_diabetes.csv", index=False)

print("\nSaved → preprocessed_diabetes.csv")


Saved → preprocessed_diabetes.csv
